In [1]:
import requests
import pandas as pd
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
API_KEY = "166E1F2D-5FD5-4063-BA75-CFA20CE6798B"

## CODIGO PARA EXTRACCION DE DATA SETS DE LICITACIONES Y ORDENES DE COMPRA

In [3]:
# ---------------------------------------
# Función consulta API con reintentos
# ---------------------------------------

def consultar_api(url, max_retries=5):

    for intento in range(max_retries):

        try:
            response = requests.get(url)

            if response.status_code == 200:
                return response.json()

            else:
                espera = (intento + 1) * 5
                print(f"Error {response.status_code}. Esperando {espera}s antes de reintentar...")
                time.sleep(espera)

        except Exception as e:
            print("Error conexión:", e)
            time.sleep(5)

    return None

In [4]:
# ---------------------------------------
# Función generar fechas por mes
# ---------------------------------------

def generar_fechas_mes(year, month):

    inicio = datetime(year, month, 1)

    if month == 12:
        fin = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        fin = datetime(year, month + 1, 1) - timedelta(days=1)

    fechas = []
    actual = inicio

    while actual <= fin:
        fechas.append(actual.strftime("%d%m%Y"))
        actual += timedelta(days=1)

    return fechas

In [ ]:
# ---------------------------------------
# Descarga LICITACIONES
# ---------------------------------------

'''licitaciones_total = []

for mes in range(1, 13):

    print("\n===========================")
    print(f"Procesando MES {mes:02d} - 2025")
    print("===========================")

    fechas = generar_fechas_mes(2025, mes)

    for i, fecha in enumerate(fechas):

        print(f"Día {i+1}/{len(fechas)} → {fecha}")

        url = f"https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json?fecha={fecha}&ticket={API_KEY}"

        data = consultar_api(url)

        if data and "Listado" in data:

            cantidad = len(data["Listado"])
            print("   registros obtenidos:", cantidad)

            licitaciones_total.extend(data["Listado"])

        else:
            print("   sin datos")

        # Espera variable para evitar bloqueo API
        espera = random.uniform(1.5, 4.0)
        time.sleep(espera)'''


print("\nTotal licitaciones descargadas:", len(licitaciones_total))

df_licitaciones = pd.DataFrame(licitaciones_total)

df_licitaciones.head()

In [ ]:
# ---------------------------------------
# Exportar
# ---------------------------------------

df_licitaciones.to_csv("licitaciones_2025.csv", index=False)
df_licitaciones.to_parquet("licitaciones_2025.parquet")

print("\nArchivo licitaciones_2025.csv generado")

In [ ]:
# ---------------------------------------
# Descarga ÓRDENES DE COMPRA
# ---------------------------------------

'''ordenes_total = []

for mes in range(1, 13):

    print("\n===========================")
    print(f"Procesando MES {mes:02d} - 2025")
    print("===========================")

    fechas = generar_fechas_mes(2025, mes)

    for i, fecha in enumerate(fechas):

        print(f"Día {i+1}/{len(fechas)} → {fecha}")

        url = f"https://api.mercadopublico.cl/servicios/v1/publico/ordenesdecompra.json?fecha={fecha}&ticket={API_KEY}"

        data = consultar_api(url)

        if data and "Listado" in data:

            cantidad = len(data["Listado"])
            print("   registros obtenidos:", cantidad)

            ordenes_total.extend(data["Listado"])

        else:
            print("   sin datos")

        # Espera aleatoria para evitar bloqueo API
        espera = random.uniform(1.5, 4.0)
        time.sleep(espera)'''


print("\nTotal órdenes de compra descargadas:", len(ordenes_total))

df_ordenes = pd.DataFrame(ordenes_total)

df_ordenes.head()

In [ ]:
# ---------------------------------------
# Exportar resultados
# ---------------------------------------

df_ordenes.to_csv("ordenes_compra_2025.csv", index=False)
df_ordenes.to_parquet("ordenes_compra_2025.parquet")

print("\nArchivo ordenes_compra_2025.csv generado")

## ALGUNAS CONSULTAS

In [2]:
df_licitaciones=pd.read_parquet("licitaciones_dataset_completo.parquet")

In [3]:
df_licitaciones.head()

,CodigoExterno,Nombre,CodigoEstado,FechaCierre
0,564162-187-L119,(id.48430) Particulas para Depto de Física,7,2020-01-01T17:04:01.527
1,1058134-381-L119,Medicamentos Enero 2020,7,2020-01-01T15:04:01.643
2,1663-111-L119,SERVICIO DE LABORATORIO DENTAL PARA PRÓTESIS M...,7,2020-01-01T21:44:01.157
3,2713-290-L119,CLASES DIRIGIDAS,7,2020-01-01T19:29:01.22
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00


In [4]:
df_licitaciones.columns

Index(['CodigoExterno', 'Nombre', 'CodigoEstado', 'FechaCierre'], dtype='object')

In [5]:
df_licitaciones.shape

(826238, 4)

In [6]:
df_licitaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 826238 entries, 0 to 826237
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CodigoExterno  826238 non-null  object
 1   Nombre         826238 non-null  object
 2   CodigoEstado   826238 non-null  int64 
 3   FechaCierre    826237 non-null  object
dtypes: int64(1), object(3)
memory usage: 25.2+ MB


In [17]:
df_licitaciones['tipo_licitacion'] = df_licitaciones['CodigoExterno'].str.extract(r'-([A-Z0-9]+)\d{2}$')

# Contamos cuántas veces aparece cada tipo y sacamos el Top 3
top_3_tipos_lic = df_licitaciones['tipo_licitacion'].value_counts().head(3)

print("Los 3 tipos de licitaciones más comunes son:")
print(top_3_tipos_lic)

Los 3 tipos de licitaciones más comunes son:
tipo_licitacion
LE    361115
L1    269241
LP     63356
Name: count, dtype: int64


In [5]:
df_licitaciones['codigo_organismo'] = df_licitaciones['CodigoExterno'].str.extract(r'^(\d+)-')

# Contamos cuántas veces aparece cada organismo y sacamos el Top 3
top_3_organismos = df_licitaciones['codigo_organismo'].value_counts().head(3)

print("Los 3 organismos más comunes son:")
print(top_3_organismos)

Los 3 organismos más comunes son:
codigo_organismo
621     7483
1658    4888
2409    4552
Name: count, dtype: int64


In [11]:
df_licitaciones.isnull().sum().sort_values(ascending=False)

tipo_licitacion    12
FechaCierre         1
CodigoExterno       0
CodigoEstado        0
Nombre              0
dtype: int64

In [12]:
df_ordenes= pd.read_parquet("ordenes_compra_dataset_completo.parquet")

In [18]:
df_ordenes.head()

,Codigo,Nombre,CodigoEstado
0,1057432-1233-SE19,ORDEN DE COMPRA DESDE 1057432-109-LP19,6
1,1058012-3776-SE19,(FAR) ESPECIAL DICIEMBRE / DICLOFENACO CM Y O...,6
2,1058012-4076-SE19,"CONTROLADOS DIC ESPECIAL/CLONAZEPAM 2,5 GTS",6
3,1499-6130-SE19,SONDA ECO/ICE,6
4,1499-6381-SE19,Servicios Octubre,6


In [19]:
df_ordenes.columns

Index(['Codigo', 'Nombre', 'CodigoEstado'], dtype='object')

In [20]:
df_ordenes.shape

(13489635, 3)

In [21]:
df_ordenes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13489635 entries, 0 to 13489634
Data columns (total 3 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   Codigo        object
 1   Nombre        object
 2   CodigoEstado  int64 
dtypes: int64(1), object(2)
memory usage: 308.8+ MB


In [18]:
df_ordenes['tipo_orden'] = df_ordenes['Codigo'].str.extract(r'-([A-Z0-9]+)\d{2}$')

# Contamos cuántas veces aparece cada tipo y sacamos el Top 3
top_3_tipos = df_ordenes['tipo_orden'].value_counts().head(3)

print("Los 3 tipos de órdenes más comunes son:")
print(top_3_tipos)

Los 3 tipos de órdenes más comunes son:
tipo_orden
SE    7015048
AG    3938660
CM    2266940
Name: count, dtype: int64


In [25]:
df_ordenes.isnull().sum().sort_values(ascending=False)

tipo_orden      8
Codigo          1
Nombre          0
CodigoEstado    0
dtype: int64

## EXTRAYENDO DETALLES PARA ENRIQUECIMIENTO DE DATA ORDENES DE COMPRA-ITEMS

In [2]:
API_KEY = "166E1F2D-5FD5-4063-BA75-CFA20CE6798B"

# Configuracion de variables a usar
MAX_WORKERS = 10 #10 hilos en paralelo, más velocidad
BATCH_SIZE = 500 #grupo de codigos escalable
FLUSH_SIZE = 2000  # cada cuántos registros guardar
mes = "2025_12"
OUTPUT_PATH = f"ordenes_detalle_{mes}.parquet"
CONTROL_PATH = f"control_codigos_{mes}.parquet"

In [3]:
# Request
def obtener_detalle(codigo):
    url = f"https://api.mercadopublico.cl/servicios/v1/publico/ordenesdecompra.json?codigo={codigo}&ticket={API_KEY}"
    
    headers = {"User-Agent": "data-pipeline/1.0"} #Presentación hacia la API

    for intento in range(3): #3 intentos (0; 1 y2)
        try:
            r = requests.get(url, timeout=10, headers=headers) #llamamos a la API. Si no responde en 10 segundos, corta la conexión y pasa al siguiente intento.

            if r.status_code != 200: #Si el servidor no responde bien, no es 200 (OK)
                time.sleep(1) #espera 1 segundo, vuelve a intentar
                continue

            data = r.json() #convierte la informacion a json

            if not data or "Listado" not in data: #si no tiene el formato esperado
                return {"Codigo": codigo, "status": "error"} #devuelve esto

            listado = data["Listado"] #creamos la variable para la carpeta listado

            if not listado: #si no hay datos en el listado
                return {"Codigo": codigo, "status": "empty"} #devuelve esto

            o = listado[0] #entra al primer grupo de campos

            return {
                "Codigo": o.get("Codigo"),
                "CodigoEstado": o.get("CodigoEstado"),
                "CodigoLicitacion": o.get("CodigoLicitacion"),
                "status": "ok"
            }

        except Exception as e:
            if intento == 2: #si el codigo falló y se encuentra en el ultimo tercer intento (0; 1 y 2)
                return {"Codigo": codigo, "status": "error"} #devuelve esto
            time.sleep(1) #si no esta en el ultimo intento que falló, espera 1 segundo y vuelve a intentar


In [4]:
# Cargar data
df = pd.read_parquet("ordenes_compra_2025_12.parquet", columns=["Codigo", "CodigoEstado"]) #cargamos la data

#Filtramos la data
df = df[
    (df["CodigoEstado"] == 6) & #Aceptadas
    (df["Codigo"].notna()) &
    (df["Codigo"].str.extract(r'-([A-Z]{2})')[0] == "SE") #usamos regex
]

codigos = df["Codigo"].drop_duplicates().tolist() #eliminamos duplicados

print("Total códigos filtrados:", len(codigos))

Total códigos filtrados: 21414


In [5]:
# Control del progreso
if os.path.exists(CONTROL_PATH): #si el archivo de CONTROL_PATH existe en mi carpeta
    df_control_old = pd.read_parquet(CONTROL_PATH) #cargamos el archivo
    procesados = set(df_control_old["Codigo"]) #convertimos a conjunto todos los codigos extraidos
else:
    df_control_old = pd.DataFrame() #si el archivo aun no existe, creamos un dataframe vacio que usaremos luego
    procesados = set() #creamos un conjunto vacio para los codigos que luego se iran procesando

codigos_pendientes = [c for c in codigos if c not in procesados] #creamos una lista con un loop que recorre codigo por codigo de la adata de codigos sin incluir lo que ya esten procesados

print("Códigos pendientes:", len(codigos_pendientes))

Códigos pendientes: 21414


In [6]:
# Pipline
buffer_data = []
buffer_control = []

for i in range(0, len(codigos_pendientes), BATCH_SIZE):
    
    batch = codigos_pendientes[i:i+BATCH_SIZE] #avanza de 500 en 500 codigos
    print(f"Procesando batch {i} - {i+len(batch)}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor: #paraleliza
        futures = {executor.submit(obtener_detalle, c): c for c in batch} #creamos diccionario

        for future in as_completed(futures):
            res = future.result()

            if not res:
                continue

            # separar control vs data
            status = res.get("status", "error")

            buffer_control.append({
                "Codigo": res.get("Codigo"),
                "status": status
            })

            if status == "ok":
                buffer_data.append(res)

    
    # Guardado controlado
    if len(buffer_data) >= FLUSH_SIZE:

        df_data = pd.DataFrame(buffer_data)
        df_control = pd.DataFrame(buffer_control)

        # guardar data
        if os.path.exists(OUTPUT_PATH):
            df_old = pd.read_parquet(OUTPUT_PATH)
            df_total = pd.concat([df_old, df_data], ignore_index=True)
        else:
            df_total = df_data

        df_total = df_total.drop_duplicates(subset=["Codigo"])
        df_total.to_parquet(OUTPUT_PATH, index=False)

        # guardar control
        if not df_control_old.empty:
            df_control_total = pd.concat([df_control_old, df_control], ignore_index=True)
        else:
            df_control_total = df_control

        df_control_total = df_control_total.drop_duplicates(subset=["Codigo"])
        df_control_total.to_parquet(CONTROL_PATH, index=False)

        print(f"Guardados {len(df_data)} registros")

        # limpiar buffers
        buffer_data = []
        buffer_control = []
        df_control_old = df_control_total

    time.sleep(1)

Procesando batch 0 - 500
Procesando batch 500 - 1000
Procesando batch 1000 - 1500
Procesando batch 1500 - 2000
Procesando batch 2000 - 2500
Procesando batch 2500 - 3000
Procesando batch 3000 - 3500
Procesando batch 3500 - 4000
Procesando batch 4000 - 4500
Procesando batch 4500 - 5000
Procesando batch 5000 - 5500
Procesando batch 5500 - 6000
Procesando batch 6000 - 6500
Procesando batch 6500 - 7000
Procesando batch 7000 - 7500
Procesando batch 7500 - 8000
Procesando batch 8000 - 8500
Procesando batch 8500 - 9000
Procesando batch 9000 - 9500
Procesando batch 9500 - 10000
Procesando batch 10000 - 10500
Procesando batch 10500 - 11000
Procesando batch 11000 - 11500
Procesando batch 11500 - 12000
Procesando batch 12000 - 12500
Procesando batch 12500 - 13000
Procesando batch 13000 - 13500
Procesando batch 13500 - 14000
Procesando batch 14000 - 14500
Procesando batch 14500 - 15000
Procesando batch 15000 - 15500
Procesando batch 15500 - 16000
Procesando batch 16000 - 16500
Procesando batch 1650

In [7]:
# Guardado Final
if buffer_data:
    df_data = pd.DataFrame(buffer_data)

    if os.path.exists(OUTPUT_PATH):
        df_old = pd.read_parquet(OUTPUT_PATH)
        df_total = pd.concat([df_old, df_data], ignore_index=True)
    else:
        df_total = df_data

    df_total = df_total.drop_duplicates(subset=["Codigo"])
    df_total.to_parquet(OUTPUT_PATH, index=False)

if buffer_control:
    df_control = pd.DataFrame(buffer_control)

    if os.path.exists(CONTROL_PATH):
        df_old = pd.read_parquet(CONTROL_PATH)
        df_control_total = pd.concat([df_old, df_control], ignore_index=True)
    else:
        df_control_total = df_control

    df_control_total = df_control_total.drop_duplicates(subset=["Codigo"])
    df_control_total.to_parquet(CONTROL_PATH, index=False)


In [8]:
# Reporte final
df_control = pd.read_parquet(CONTROL_PATH)

print("\nResumen final:")
print(df_control.groupby("status").size())


Resumen final:
status
empty       2
error     226
ok       2310
dtype: int64


In [20]:
df_total.head(20)

,Codigo,CodigoEstado,CodigoLicitacion,status
0,1050283-3108-SE25,6,1050283-7-LQ25,ok
1,1057429-7573-SE25,6,,ok
2,1057501-32210-SE25,6,1057501-112-LP25,ok
3,1058095-3055-SE25,6,1058095-90-LR24,ok
4,1075963-48918-SE25,6,1075963-314-LE25,ok
5,1057500-106-SE25,6,,ok
6,1057494-9483-SE25,6,1058090-16-LP25,ok
7,1176-125-SE25,6,1176-1-LP25,ok
8,1057480-14117-SE25,6,1057480-10-LE25,ok
9,1214102-2150-SE25,6,1214102-18-LP25,ok


In [19]:
len(df_total)

2310

## EXTRAYENDO ORGANISMOS

In [4]:
def consulta_organismos(intentos = 3):
    url_organismos = f"https://api.mercadopublico.cl/servicios/v1/Publico/Empresas/BuscarComprador?ticket={API_KEY}"
    for intento in range (intentos):
        try:
            response = requests.get(url_organismos, timeout=10)

            if response.status_code == 200:
                data = response.json()
                if not data or "listaEmpresas" not in data:
                    print ("Respuesta sin datos validos")
                    return None
                return data["listaEmpresas"]

            else:
                espera = (intento + 1) * 5
                print(f"Error {response.status_code}. Esperando {espera}s antes de reintentar...")
                time.sleep(espera)
                
        except Exception as e:
            print("Error conexión: ",e)
            time.sleep(5)
    return None    

In [5]:
def transformar_organismos(data):
    organismos = []
    for o in data:
        organismos.append({
            "CodigoOrganismo": o.get("CodigoEmpresa"),
            "NombreOrganismo": o.get("NombreEmpresa")
        })
    return organismos

In [7]:
data = consulta_organismos()
if data:
    lista = transformar_organismos(data)
    df_organismos = pd.DataFrame(lista)
    df_organismos.to_parquet("organismos.parquet", index= False)
    print("Total de organismos: ", len(df_organismos))

Total de organismos:  899


In [8]:
df_organismos.head()

,CodigoOrganismo,NombreOrganismo
0,7086,AGENCIA CHILENA DE COOPERACION INTERNACIONAL P...
1,1224636,AGENCIA CHILENA DE EFICIENCIA ENERGETICA
2,7193,AGENCIA DE PROMOCION DE LA INVERSION EXTRANJERA
3,7212,Agencia Nacional de Inteligencia
4,1824441,ASOCIACION DE MUNICIPIOS METROPOLITANOS PARA L...


In [11]:
df_organismos.isnull().sum()

CodigoOrganismo    0
NombreOrganismo    0
dtype: int64

# EXTRAYENDO DETALLES DE LICITACIONES

In [22]:
!pip install google-cloud-storage

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --------------------- ------------------ 1.8/3.5 MB 9.1 MB/s eta 0:00:01
   --------------------------------- ------ 2.9/3.5 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 6.2 MB/s  0:00:00

   ----------------------------------------  0/13 [pyasn1]
   ----------------------------------------  0/13 [pyasn1]
   ----------------------------------------  0/13 [pyasn1]
   ----------------------------------------  0/13 [pyasn1]
   ----------------------------------------  0/13 [pyasn1]
   --- ------------------------------------  1/13 [protobuf]
   --- ------------------------------------  1/13 [protobuf]
   --- ------------------------------------  1/13 [protobuf]
   --- ------------------------------------  1/13 [protobuf]
   --- ------------------------------------  1/13 [protobuf]
   --- --------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\PREDATOR\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
import os
import io
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import storage

# CREDENCIAL
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/PREDATOR/gcp/proyecto-mercadopublico-71d58797d1f1.json"

# BUCKET
BUCKET_NAME = "mp_datos_proyecto"
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

# API
API_KEY = "166E1F2D-5FD5-4063-BA75-CFA20CE6798B"

# PARÁMETROS
YEAR = 2022  # prueba solo 1 año
SAMPLE_SIZE = 50
MAX_WORKERS = 2
BATCH_SIZE = 50

# PATHS
BASE_PATH = "data/licitaciones/processed/anual"
OUTPUT_PATH = "data/licitaciones/staging/mapping_codigo_organismo"

In [23]:
def leer_parquet_anual(year):
    file_path = f"{BASE_PATH}/licitaciones_{year}.parquet"
    
    blob = bucket.blob(file_path)

    if not blob.exists():
        print(f"No existe {file_path}")
        return pd.DataFrame()

    data = blob.download_as_bytes()
    df = pd.read_parquet(io.BytesIO(data))

    print(f"Filas cargadas: {len(df)}")

    return df

In [24]:
def obtener_detalle(codigo):
    url = f"https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json?codigo={codigo}&ticket={API_KEY}"

    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"Error HTTP {r.status_code} para codigo {codigo}")
            return {"Codigo": codigo, "Error": f"HTTP {r.status_code}"}

        data = r.json()
        listado = data.get("Listado", [])

        if not listado:
            # Licitación sin datos → no eliminar el código, solo marcar
            return {"Codigo": codigo, "Error": "Listado vacío"}

        lic = listado[0]
        items = lic.get("Items", {}).get("Listado", [])

        return {
            "Codigo": lic.get("CodigoExterno", codigo),
            "Tipo": lic.get("Tipo"),
            "CodigoOrganismo": lic.get("Comprador", {}).get("CodigoOrganismo"),
            "RegionUnidad": lic.get("Comprador", {}).get("RegionUnidad"),
            "CantidadItems": len(items)
        }

    except Exception as e:
        print(f"Exception para {codigo}: {e}")
        return {"Codigo": codigo, "Error": str(e)}

In [25]:
def subir_parquet_gcp(df, path):
    temp_file = "temp.parquet"
    df.to_parquet(temp_file, index=False)

    blob = bucket.blob(path)
    blob.upload_from_filename(temp_file)

    os.remove(temp_file)

In [26]:
print(f"\n===== PROCESANDO AÑO {YEAR} =====")

df = leer_parquet_anual(YEAR)

if df.empty:
    print("Dataset vacío")
else:

    # FILTRO BASE (SIN tipo todavía)
    df = df[
        (df["CodigoEstado"] == 8) &
        (df["CodigoExterno"].notna())
    ]

    print("Filas después de filtro:", len(df))

    # MUESTREO INTELIGENTE
    df_sample = df.sample(SAMPLE_SIZE, random_state=42)

    codigos = df_sample["CodigoExterno"].tolist()

    print("Códigos a procesar:", len(codigos))

    resultados = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(obtener_detalle, c): c for c in codigos}

        ok_count = 0
        error_count = 0

        for future in as_completed(futures):
            res = future.result()
            if res.get("Error") is None:
                resultados.append(res)
                ok_count += 1
            else:
                error_count += 1

    print(f"\nRESULTADO → OK: {ok_count} | CON ERROR: {error_count}")

    # CONVERTIR A DF
    df_out = pd.DataFrame(resultados)

    if not df_out.empty:

        print("\nTipos encontrados:")
        print(df_out["Tipo"].value_counts())

        # FILTRO FINAL LE
        df_out = df_out[df_out["Tipo"].notna()]
        df_le = df_out[df_out["Tipo"] == "LE"]

        print(f"\nRegistros LE: {len(df_le)}")

        # GUARDAR SOLO SI HAY DATA
        if not df_le.empty:
            output_path = f"{OUTPUT_PATH}/year={YEAR}/sample_le.parquet"
            subir_parquet_gcp(df_le, output_path)

            print(f"\nDatos guardados en: {output_path}")
        else:
            print("\n No se encontraron registros LE")
    else:
        print("\n No hubo resultados válidos de la API")


===== PROCESANDO AÑO 2022 =====
Filas cargadas: 151006
Filas después de filtro: 107605
Códigos a procesar: 50
Error HTTP 500 para codigo 1217466-46-L122
Exception para 2927-54-LE22: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 867990-150-L122: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 2446-214-LE22: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 731-13-LE22: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 2281-267-L122: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 1087105-3-LE22: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read timed out. (read timeout=30)
Exception para 951405-6-LE22: HTTPSConnectionPool(host='api.mercadopublico.cl', port=443): Read 

## PROBANDO OTRA ESTRATEGIA PARA EXTRACCION DE LICITACIONES

In [28]:
!pip install tenacity

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\PREDATOR\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
"""
pipeline_organismos_le.py  v5 — EXTRACCIÓN FOCALIZADA POR CÓDIGO
=================================================================
 
CONTEXTO:
    Los primeros números del CodigoExterno (prefijo) NO corresponden
    al CodigoOrganismo del campo Comprador. Son identificadores distintos.
    El único endpoint que devuelve CodigoOrganismo + NombreOrganismo +
    RegionUnidad de forma confiable es ?codigo=XXX (detalle por licitación).
 
ESTRATEGIA:
    - 19 prefijos de interés ya filtrados (adj. + LE + >= 150 licitaciones)
    - Por cada prefijo: tomar 1 código LE adjudicado representativo desde GCS
    - Consultar ?codigo=XXX → extraer Comprador completo (19 requests total)
    - Cruzar con conteo de licitaciones LE adjudicadas por prefijo
    - Guardar curated + audit en GCS
 
FLUJO:
    GCS parquets anuales
        → filtrar por prefijos de interés + CodigoEstado==8 + tipo LE
        → 1 código representativo por prefijo
        → 19 requests ?codigo=XXX → CodigoOrganismo, NombreOrganismo, RegionUnidad
        → JOIN con conteo por prefijo
        → curated + audit → GCS
"""
 
import os
import io
import re
import time
import logging
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import date
from google.cloud import storage
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

In [8]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
 
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "C:/Users/PREDATOR/gcp/proyecto-mercadopublico-71d58797d1f1.json"
)
 
API_KEY     = "166E1F2D-5FD5-4063-BA75-CFA20CE6798B"
BUCKET_NAME = "mp_datos_proyecto"
 
# Paths GCS
LICITACIONES_ANUAL_PATH = "data/licitaciones/processed/anual"
CURATED_PATH            = "data/licitaciones/curated/organismos_le"
AUDIT_PATH              = "data/licitaciones/audit/organismos_le"
 
# Parámetros
YEARS       = list(range(2020, 2026))
TIMEOUT_SEC = 20
SLEEP_SEC   = 1.5   # pausa entre requests (19 en total, sin prisa)
 
# Prefijos de interés: resultado de aplicar filtros
# (adjudicadas + tipo LE + >= 150 licitaciones)
# Valor del dict = ranking por volumen (referencia, no se usa en el código)
PREFIJOS_INTERES = {
    '3510', '3863', '4857', '3656', '1660', '3960', '3794',
    '2196', '979',  '3589', '2564', '2342', '4809', '1247197',
    '1057503', '3928', '3709', '1509', '3508'
}

In [9]:
# ─────────────────────────────────────────────
# GCS CLIENT
# ─────────────────────────────────────────────
 
gcs_client = storage.Client()
bucket     = gcs_client.bucket(BUCKET_NAME)

In [10]:
# ─────────────────────────────────────────────
# UTILIDADES GCS
# ─────────────────────────────────────────────
 
def leer_parquet_gcs(gcs_path: str) -> pd.DataFrame:
    blob = bucket.blob(gcs_path)
    if not blob.exists():
        log.warning(f"No existe en GCS: {gcs_path}")
        return pd.DataFrame()
    data = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(data))
 
 
def upload_parquet_gcs(df: pd.DataFrame, gcs_path: str) -> None:
    """Sube DataFrame a GCS como Parquet Snappy sin escribir en disco."""
    table  = pa.Table.from_pandas(df, preserve_index=False)
    buffer = io.BytesIO()
    pq.write_table(table, buffer, compression="snappy")
    buffer.seek(0)
    blob = bucket.blob(gcs_path)
    blob.upload_from_file(buffer, content_type="application/octet-stream")
    log.info(f"Subido: gs://{BUCKET_NAME}/{gcs_path} ({len(df)} filas)")

In [11]:
# ─────────────────────────────────────────────
# PASO 1: PREPARAR DATASET DESDE GCS
# ─────────────────────────────────────────────
 
def extraer_tipo(codigo: str):
    """
    Extrae el tipo de licitación desde CodigoExterno.
    '1658-257-LE22' → 'LE'
    '2704-190-L121' → 'L1'
    """
    match = re.search(r"-([A-Z][A-Z0-9]{0,2})\d{2}$", str(codigo))
    return match.group(1) if match else None
 
 
def extraer_prefijo(codigo: str):
    """
    Extrae los números antes del primer guión.
    '1509-5-L114' → '1509'
    '1247197-32-LE22' → '1247197'
    """
    match = re.match(r"^(\d+)-", str(codigo))
    return match.group(1) if match else None
 
 
def construir_dataset_filtrado() -> pd.DataFrame:
    """
    Lee parquets anuales de GCS y retorna solo las licitaciones
    LE adjudicadas correspondientes a los 19 prefijos de interés.
 
    Returns:
        DataFrame con columnas: CodigoExterno, prefijo, tipo, total_le_adjudicadas
    """
    log.info("── PASO 1: Cargando y filtrando dataset desde GCS ──")
 
    frames = []
    for year in YEARS:
        path = f"{LICITACIONES_ANUAL_PATH}/licitaciones_{year}.parquet"
        df   = leer_parquet_gcs(path)
        if not df.empty:
            log.info(f"  {year}: {len(df):,} filas")
            frames.append(df)
 
    if not frames:
        raise RuntimeError("Sin datasets anuales en GCS.")
 
    df_total = pd.concat(frames, ignore_index=True)
    log.info(f"Total combinado: {len(df_total):,} licitaciones")
 
    # Extraer prefijo y tipo desde CodigoExterno
    df_total["prefijo"] = df_total["CodigoExterno"].apply(extraer_prefijo)
    df_total["tipo"]    = df_total["CodigoExterno"].apply(extraer_tipo)
 
    # Filtros: adjudicadas + tipo LE + prefijos de interés
    df_filtrado = df_total[
        (df_total["CodigoEstado"] == 8) &
        (df_total["tipo"] == "LE") &
        (df_total["prefijo"].isin(PREFIJOS_INTERES))
    ].copy()
 
    log.info(f"Licitaciones LE adjudicadas en prefijos de interés: {len(df_filtrado):,}")
 
    # Conteo por prefijo
    conteo = (
        df_filtrado.groupby("prefijo")
        .size()
        .reset_index(name="total_le_adjudicadas")
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )
 
    log.info(f"\nConteo por prefijo:\n{conteo.to_string(index=False)}\n")
 
    # Un código representativo por prefijo para consultar la API
    # Tomamos el más reciente (último año disponible) para maximizar
    # probabilidad de respuesta exitosa en el endpoint de detalle
    representativos = (
        df_filtrado.sort_values("FechaCierre", ascending=False)
        .groupby("prefijo")["CodigoExterno"]
        .first()
        .reset_index()
        .rename(columns={"CodigoExterno": "codigo_representativo"})
    )
 
    log.info(f"Códigos representativos seleccionados:\n{representativos.to_string(index=False)}\n")
 
    # Unir conteo con representativo
    resultado = conteo.merge(representativos, on="prefijo", how="left")
 
    return resultado

In [12]:
# ─────────────────────────────────────────────
# PASO 2: CONSULTAR API POR CÓDIGO INDIVIDUAL
#
# ?codigo=XXX es el endpoint más confiable de la API.
# Devuelve el objeto Comprador completo con:
#   - CodigoOrganismo
#   - NombreOrganismo
#   - RegionUnidad
# Son exactamente 19 requests — completamente manejable.
# ─────────────────────────────────────────────
 
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=2, min=3, max=30),
    retry=retry_if_exception_type(
        (requests.exceptions.Timeout, requests.exceptions.ConnectionError)
    ),
    before_sleep=before_sleep_log(log, logging.WARNING),
    reraise=False,
)
def _fetch_detalle(codigo: str) -> dict | None:
    """Consulta el detalle de una licitación por código."""
    url    = "https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json"
    params = {"codigo": codigo, "ticket": API_KEY}
    r      = requests.get(url, params=params, timeout=TIMEOUT_SEC)
 
    if 400 <= r.status_code < 500:
        log.error(f"HTTP {r.status_code} para codigo={codigo} (no reintentable)")
        return None
 
    r.raise_for_status()
    return r.json()
 
 
def extraer_comprador(codigo: str, prefijo: str) -> dict:
    """
    Consulta ?codigo=XXX y extrae CodigoOrganismo, NombreOrganismo
    y RegionUnidad desde el campo Comprador.
 
    Args:
        codigo:  CodigoExterno representativo del prefijo
        prefijo: prefijo numérico del organismo
 
    Returns:
        dict auditado con los datos del comprador y status
    """
    base = {
        "prefijo":          prefijo,
        "codigo_consultado": codigo,
        "codigo_organismo": None,
        "nombre_organismo": None,
        "region_unidad":    None,
    }
 
    try:
        data = _fetch_detalle(codigo)
    except Exception as e:
        return {**base, "status": "error_timeout", "detalle_error": str(e)[:200]}
 
    if data is None:
        return {**base, "status": "error_http4xx", "detalle_error": "HTTP 4xx"}
 
    listado = data.get("Listado", [])
 
    if not listado:
        return {**base, "status": "vacio", "detalle_error": f"Listado vacío para codigo={codigo}"}
 
    # Extraer Comprador del primer registro
    comprador = listado[0].get("Comprador") or {}
 
    codigo_org = comprador.get("CodigoOrganismo")
    nombre_org = comprador.get("NombreOrganismo")
    region     = comprador.get("RegionUnidad")
 
    if not any([codigo_org, nombre_org, region]):
        return {**base, "status": "comprador_vacio", "detalle_error": "Comprador sin campos"}
 
    return {
        **base,
        "codigo_organismo": str(codigo_org).strip() if codigo_org else None,
        "nombre_organismo": str(nombre_org).strip() if nombre_org else None,
        "region_unidad":    str(region).strip()     if region     else None,
        "status":           "ok",
        "detalle_error":    None,
    }

In [13]:
# ─────────────────────────────────────────────
# PASO 3: CONSULTAR LOS 19 PREFIJOS
# ─────────────────────────────────────────────
 
def enriquecer_prefijos(df_dataset: pd.DataFrame) -> pd.DataFrame:
    """
    Itera sobre los 19 prefijos, consulta 1 código por prefijo
    y enriquece con CodigoOrganismo, NombreOrganismo y RegionUnidad.
 
    Returns:
        DataFrame enriquecido con todos los campos y status auditado.
    """
    log.info("── PASO 2: Consultando API — 1 request por prefijo (19 total) ──")
 
    resultados = []
    total = len(df_dataset)
 
    for i, row in enumerate(df_dataset.itertuples(), 1):
        log.info(
            f"  [{i:2d}/{total}] Prefijo {row.prefijo} | "
            f"código: {row.codigo_representativo}"
        )
 
        resultado = extraer_comprador(row.codigo_representativo, row.prefijo)
 
        icono = "OK" if resultado["status"] == "ok" else "X"
        log.info(
            f"          [{icono}] {resultado['status']} | "
            f"{(resultado['nombre_organismo'] or '—')[:50]} | "
            f"{(resultado['region_unidad'] or '—')[:40]}"
        )
 
        resultados.append(resultado)
        time.sleep(SLEEP_SEC)
 
    df_resultados = pd.DataFrame(resultados)
 
    # Unir con conteo
    df_merged = df_dataset[["prefijo", "total_le_adjudicadas"]].merge(
        df_resultados, on="prefijo", how="left"
    )
 
    # Resumen
    resumen = df_merged["status"].value_counts().to_dict()
    log.info(f"\nResumen consultas API:")
    for status, count in sorted(resumen.items(), key=lambda x: -x[1]):
        log.info(f"  {status:<25}: {count}")
 
    return df_merged

In [14]:
# ─────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ─────────────────────────────────────────────
 
def run_pipeline(dry_run: bool = False) -> pd.DataFrame:
    """
    Ejecuta el pipeline completo.
 
    Args:
        dry_run: Si True, ejecuta todo pero no escribe en GCS.
 
    Returns:
        DataFrame curated con los 19 organismos enriquecidos.
    """
    log.info("═" * 60)
    log.info("PIPELINE: Enriquecimiento organismos LE — 19 prefijos")
    log.info("═" * 60)
 
    # ── PASO 1: Dataset filtrado desde GCS ──────────────────────────────────
    df_dataset = construir_dataset_filtrado()
 
    if df_dataset.empty:
        log.warning("Sin datos. Verificar GCS y filtros.")
        return pd.DataFrame()
 
    # ── PASO 2: Enriquecimiento desde API (19 requests) ─────────────────────
    df_enriquecido = enriquecer_prefijos(df_dataset)
 
    # ── Separar capas ────────────────────────────────────────────────────────
 
    # Curated: registros con nombre y región resueltos
    df_curated = (
        df_enriquecido[df_enriquecido["status"] == "ok"][[
            "prefijo",
            "codigo_organismo",
            "nombre_organismo",
            "region_unidad",
            "total_le_adjudicadas",
        ]]
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )
 
    # Audit: todos los prefijos con trazabilidad completa
    df_audit = (
        df_enriquecido[[
            "prefijo",
            "codigo_consultado",
            "codigo_organismo",
            "nombre_organismo",
            "region_unidad",
            "total_le_adjudicadas",
            "status",
            "detalle_error",
        ]]
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )
 
    # ── Preview ──────────────────────────────────────────────────────────────
    log.info(f"\n{'═'*60}")
    log.info("RESULTADO CURATED:")
    log.info(f"{'═'*60}")
    log.info(f"\n{df_curated.to_string(index=False)}\n")
 
    # ── Guardar en GCS ───────────────────────────────────────────────────────
    if not dry_run:
        run_date = date.today().strftime("%Y-%m-%d")
 
        upload_parquet_gcs(
            df_curated,
            f"{CURATED_PATH}/fecha={run_date}/organismos_le.parquet"
        )
        upload_parquet_gcs(
            df_audit,
            f"{AUDIT_PATH}/fecha={run_date}/audit_organismos_le.parquet"
        )
        log.info("Datos guardados en GCS correctamente.")
    else:
        log.info("[DRY RUN] No se escribió en GCS.")
 
    log.info(f"\nPipeline completado: {len(df_curated)}/{len(df_enriquecido)} organismos en curated.")
    return df_curated

In [15]:
# ─────────────────────────────────────────────
# ENTRYPOINT
# ─────────────────────────────────────────────
 
if __name__ == "__main__":
    df_final = run_pipeline(dry_run=False)

04:08:04 [INFO] ════════════════════════════════════════════════════════════
04:08:04 [INFO] PIPELINE: Enriquecimiento organismos LE — 19 prefijos
04:08:04 [INFO] ════════════════════════════════════════════════════════════
04:08:04 [INFO] ── PASO 1: Cargando y filtrando dataset desde GCS ──
04:08:30 [INFO]   2020: 104,527 filas
04:08:52 [INFO]   2021: 127,624 filas
04:09:12 [INFO]   2022: 151,006 filas
04:09:20 [INFO]   2023: 160,789 filas
04:09:23 [INFO]   2024: 168,818 filas
04:09:27 [INFO]   2025: 113,548 filas
04:09:27 [INFO] Total combinado: 826,312 licitaciones
04:09:30 [INFO] Licitaciones LE adjudicadas en prefijos de interés: 4,083
04:09:30 [INFO] 
Conteo por prefijo:
prefijo  total_le_adjudicadas
   3656                   495
   3794                   284
   1660                   279
   3863                   254
   4809                   248
1247197                   220
   3709                   207
    979                   207
   3510                   204
   1509       

In [16]:
df_final.head()

,prefijo,codigo_organismo,nombre_organismo,region_unidad,total_le_adjudicadas
0,3656,116215,I MUNICIPALIDAD DE REQUINOA,Región del Libertador General Bernardo O´Higgins,495
1,3794,113812,DIRECCION DE LOGISTICA DE CARABINEROS,Región Metropolitana de Santiago,284
2,1660,7323,SERVICIO SALUD COQUIMBO HOSP DE SALAMANC,Región de Coquimbo,279
3,3863,117584,I MUNICIPALIDAD DE SANTA CRUZ,Región del Libertador General Bernardo O´Higgins,254
4,4809,7248,MINISTERIO DE OBRAS PUBLICAS DIREC CION GRAL D...,Región de la Araucanía,248


In [18]:
import io
import pandas as pd
from google.cloud import storage

# ── Configuración ────────────────────────────────────────
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "C:/Users/PREDATOR/gcp/proyecto-mercadopublico-71d58797d1f1.json"
)

BUCKET_NAME = "mp_datos_proyecto"
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

# ── Leer parquet desde GCS ───────────────────────────────
from datetime import date
run_date = "2026-04-12"  # ajusta a la fecha en que ejecutaste el pipeline

parquet_path = f"data/licitaciones/curated/organismos_le/fecha={run_date}/organismos_le.parquet"

blob = bucket.blob(parquet_path)
df_organismos = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))

print(f"Filas cargadas: {len(df_organismos)}")
display(df_organismos.head())

# ── Convertir a CSV y subir a GCS ───────────────────────
csv_buffer = io.StringIO()
df_organismos.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

csv_path = f"data/licitaciones/curated/organismos_le/fecha={run_date}/organismos_le.csv"
bucket.blob(csv_path).upload_from_string(
    csv_buffer.getvalue(),
    content_type="text/csv"
)

print(f"CSV subido a GCS: {csv_path}")

Filas cargadas: 19


,prefijo,codigo_organismo,nombre_organismo,region_unidad,total_le_adjudicadas
0,3656,116215,I MUNICIPALIDAD DE REQUINOA,Región del Libertador General Bernardo O´Higgins,495
1,3794,113812,DIRECCION DE LOGISTICA DE CARABINEROS,Región Metropolitana de Santiago,284
2,1660,7323,SERVICIO SALUD COQUIMBO HOSP DE SALAMANC,Región de Coquimbo,279
3,3863,117584,I MUNICIPALIDAD DE SANTA CRUZ,Región del Libertador General Bernardo O´Higgins,254
4,4809,7248,MINISTERIO DE OBRAS PUBLICAS DIREC CION GRAL D...,Región de la Araucanía,248


CSV subido a GCS: data/licitaciones/curated/organismos_le/fecha=2026-04-12/organismos_le.csv


In [5]:
# ── Listar todos los archivos en el bucket bajo organismos_le ────────────
blobs = list(client.list_blobs(BUCKET_NAME, prefix="data/licitaciones/curated/organismos_le"))

if blobs:
    print("Archivos encontrados:")
    for b in blobs:
        print(" ", b.name)
else:
    print("No se encontró nada en ese path.")

Archivos encontrados:
  data/licitaciones/curated/organismos_le/fecha=2026-03-28/organismos_le.parquet
  data/licitaciones/curated/organismos_le/fecha=2026-03-30/organismos_le.parquet
